# UTA-RLDD → per-frame fatigue features (Colab)

Processes RLDD **one zip at a time**: copy from your Drive shortcut → unzip → MediaPipe face features (pretrained, no training) → save a small CSV per video to Drive → delete the video.
111 GB of video becomes a few hundred MB of features. **Resumable:** rerun any time; finished videos and zips are skipped.

**Before running**
1. `Drowsiness` RLDD folder added as a shortcut in *My Drive* (done).
2. Upload `dms_code.zip` (from the project's `dist/` folder) to *My Drive/DMS/code/*.
3. Runtime: CPU is enough (MediaPipe runs on CPU). Keep the tab open. About 3 h per fold (2 videos at a time).

In [ ]:
# 1. Settings
RLDD_DIR   = '/content/drive/MyDrive/Drowsiness'          # your RLDD shortcut
OUT_DIR    = '/content/drive/MyDrive/DMS/processed/rldd'  # small feature CSVs (kept)
CODE_ZIP   = '/content/drive/MyDrive/DMS/code/dms_code.zip'
FOLDS      = ['Fold1']   # start with Fold1; later ['Fold1','Fold2','Fold3','Fold4','Fold5']
TARGET_FPS = 15          # 15 fps still resolves blinks (100-400 ms) and halves the time
WORK       = '/content/work'

In [ ]:
# 2. Mount Drive and check the RLDD files
from google.colab import drive
drive.mount('/content/drive')
import glob, os
zips = sorted(glob.glob(f'{RLDD_DIR}/*.zip'))
for z in zips:
    print(f'{os.path.getsize(z)/1e9:6.2f} GB  {os.path.basename(z)}')
assert zips, 'No zips found - check RLDD_DIR'

In [ ]:
# 3. Install the project code and the pretrained face model
!rm -rf /content/dms && mkdir -p /content/dms && unzip -q -o "$CODE_ZIP" -d /content/dms
%pip install -q -e /content/dms
!mkdir -p /content/dms/models && curl -sL -o /content/dms/models/face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
import sys
# /content/dms itself would shadow the dms package (Colab's working dir is /content): drop any cached copy
for k in [k for k in sys.modules if k == 'dms' or k.startswith('dms.')]:
    del sys.modules[k]
sys.path[:0] = ['/content/dms/src', '/content/dms/scripts']
from extract_cabin_features import extract
print('ready')

In [ ]:
# 4. Process: one zip at a time, 2 videos in parallel (Colab has 2 CPU cores), resumable
import shutil, subprocess, time, os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

VIDEO_EXT = {'.mov', '.mp4', '.m4v', '.avi', '.mkv', '.wmv'}
LABELS = {'0': 'alert', '5': 'low_vigilance', '10': 'drowsy'}

def run_one(v, out):
    t1 = time.time()
    df = extract(Path(v), Path(out), progress=False, target_fps=TARGET_FPS)
    return f'  {Path(out).name}: {len(df)} frames, face {df.face.mean():.0%}, {time.time()-t1:.0f}s'

os.makedirs(OUT_DIR, exist_ok=True)
for z in [z for z in zips if any(os.path.basename(z).startswith(f + '_') for f in FOLDS)]:
    name = Path(z).stem                      # e.g. Fold1_part1
    done = Path(OUT_DIR) / f'{name}.done'
    if done.exists():
        print(f'skip {name} (done)'); continue
    shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)
    t0 = time.time(); print(f'copying {name} ...', flush=True)
    shutil.copy(z, f'{WORK}/{name}.zip')
    subprocess.run(['unzip', '-q', '-o', f'{WORK}/{name}.zip', '-d', f'{WORK}/x'], check=True)
    os.remove(f'{WORK}/{name}.zip')
    jobs = []
    for v in sorted(p for p in Path(f'{WORK}/x').rglob('*') if p.suffix.lower() in VIDEO_EXT):
        out = Path(OUT_DIR) / name.split('_')[0] / f'{v.parent.name}_{LABELS.get(v.stem, "unknown_" + v.stem)}.csv'
        if not out.exists():
            jobs.append((str(v), str(out)))
    print(f'  {len(jobs)} videos left', flush=True)
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
        for f in as_completed([ex.submit(run_one, v, o) for v, o in jobs]):
            print(f.result(), flush=True)
    shutil.rmtree(WORK, ignore_errors=True)
    done.touch()
    print(f'{name} finished in {(time.time()-t0)/60:.1f} min')


In [ ]:
# 5. Summary of what is on Drive
import pandas as pd
rows = [{'fold': p.parent.name, 'file': p.name, 'label': p.stem.split('_', 1)[1]}
        for p in Path(OUT_DIR).rglob('*.csv')]
s = pd.DataFrame(rows)
print(s.groupby(['fold', 'label']).size().unstack(fill_value=0) if len(s) else 'nothing yet')